In [2]:
!pip install sqlalchemy pyodbc

  Using cached sqlalchemy-2.0.49-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached pyodbc-5.3.0-cp314-cp314-win_amd64.whl.metadata (2.8 kB)
  Using cached greenlet-3.5.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached sqlalchemy-2.0.49-cp314-cp314-win_amd64.whl (2.1 MB)
Using cached pyodbc-5.3.0-cp314-cp314-win_amd64.whl (72 kB)
Using cached greenlet-3.5.0-cp314-cp314-win_amd64.whl (239 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   -------------------- ------------------- 2/4 [greenlet]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ---------------------


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
from sqlalchemy import create_engine, text

df = pd.read_csv('final_cleaned_real_estate.csv')

connection_string = "mssql+pyodbc://localhost/RealEstate_DWH?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

print("Dropping old tables if exist and creating new Star Schema with Relations...")

sql_schema = """
DROP TABLE IF EXISTS fact_real_estate;
DROP TABLE IF EXISTS dim_locations;
DROP TABLE IF EXISTS dim_properties;
DROP TABLE IF EXISTS dim_payments;
DROP TABLE IF EXISTS dim_date;

CREATE TABLE dim_locations (
    Location_ID INT NOT NULL PRIMARY KEY,
    governorate NVARCHAR(255),
    city NVARCHAR(255)
);

CREATE TABLE dim_properties (
    Property_Type_ID INT NOT NULL PRIMARY KEY,
    type NVARCHAR(255),
    has_maid NVARCHAR(50),
    is_studio NVARCHAR(50)
);

CREATE TABLE dim_payments (
    Payment_ID INT NOT NULL PRIMARY KEY,
    payment_method NVARCHAR(255)
);

CREATE TABLE dim_date (
    Date DATETIME NOT NULL PRIMARY KEY,
    Year INT,
    Quarter INT,
    Month INT,
    MonthName NVARCHAR(50),
    Day INT,
    DayOfWeek INT
);

CREATE TABLE fact_real_estate (
    price FLOAT,
    size_sqm FLOAT,
    bedrooms_count INT,
    bathrooms_count INT,
    available_from DATETIME,
    Location_ID INT,
    Property_Type_ID INT,
    Payment_ID INT,
    CONSTRAINT FK_Fact_Locations FOREIGN KEY (Location_ID) REFERENCES dim_locations(Location_ID),
    CONSTRAINT FK_Fact_Properties FOREIGN KEY (Property_Type_ID) REFERENCES dim_properties(Property_Type_ID),
    CONSTRAINT FK_Fact_Payments FOREIGN KEY (Payment_ID) REFERENCES dim_payments(Payment_ID),
    CONSTRAINT FK_Fact_Date FOREIGN KEY (available_from) REFERENCES dim_date(Date)
);
"""

with engine.connect() as conn:
    conn.execute(text(sql_schema))
    conn.commit()

print("Processing data transformation...")

try:
    dim_locations = df[['governorate', 'city']].drop_duplicates().reset_index(drop=True)
    dim_locations['Location_ID'] = dim_locations.index + 1
    dim_locations = dim_locations[['Location_ID', 'governorate', 'city']]
    
    dim_properties = df[['type', 'has_maid', 'is_studio']].drop_duplicates().reset_index(drop=True)
    dim_properties['Property_Type_ID'] = dim_properties.index + 1
    dim_properties = dim_properties[['Property_Type_ID', 'type', 'has_maid', 'is_studio']]
    
    dim_payments = df[['payment_method']].drop_duplicates().reset_index(drop=True)
    dim_payments['Payment_ID'] = dim_payments.index + 1
    dim_payments = dim_payments[['Payment_ID', 'payment_method']]
    
    df['available_from'] = pd.to_datetime(df['available_from'])
    unique_dates = pd.DataFrame({'Date': df['available_from'].dropna().unique()})
    dim_date = pd.DataFrame()
    dim_date['Date'] = unique_dates['Date']
    dim_date['Year'] = dim_date['Date'].dt.year
    dim_date['Quarter'] = dim_date['Date'].dt.quarter
    dim_date['Month'] = dim_date['Date'].dt.month
    dim_date['MonthName'] = dim_date['Date'].dt.strftime('%B')
    dim_date['Day'] = dim_date['Date'].dt.day
    dim_date['DayOfWeek'] = dim_date['Date'].dt.dayofweek
    
    fact_df = df.merge(dim_locations, on=['governorate', 'city'], how='left')
    fact_df = fact_df.merge(dim_properties, on=['type', 'has_maid', 'is_studio'], how='left')
    fact_df = fact_df.merge(dim_payments, on=['payment_method'], how='left')
    
    fact_real_estate = fact_df[[
        'price', 'size_sqm', 'bedrooms_count', 'bathrooms_count', 
        'available_from', 'Location_ID', 'Property_Type_ID', 'Payment_ID'
    ]]
    
    print("Uploading data to linked tables...")
    
    dim_locations.to_sql('dim_locations', con=engine, if_exists='append', index=False)
    dim_properties.to_sql('dim_properties', con=engine, if_exists='append', index=False)
    dim_payments.to_sql('dim_payments', con=engine, if_exists='append', index=False)
    dim_date.to_sql('dim_date', con=engine, if_exists='append', index=False)
    fact_real_estate.to_sql('fact_real_estate', con=engine, if_exists='append', index=False)
    
    print("Success! Schema created and data uploaded successfully.")

except Exception as e:
    print("Error occurred:")
    print(e)

Dropping old tables if exist and creating new Star Schema with Relations...
Processing data transformation...
Uploading data to linked tables...
Success! Schema created and data uploaded successfully.
